# 🤖 ego2dex on Google Colab

**Egocentric video → 3D hand annotations → dexterous robot hand retargeting**

This notebook walks you through:
1. ✅ Installing ego2dex (core + GPU models)
2. ✅ Running the smoke test (CPU, no weights)
3. ✅ Installing real GPU models (HaMeR, Grounding DINO, SAM2, etc.)
4. ✅ Processing YOUR egocentric videos
5. ✅ Exporting training-ready data (COCO / HDF5 / JSON)

> ⚠️ **Make sure you select a GPU runtime!**  
> Go to `Runtime → Change runtime type → T4 GPU` (free tier) or `A100/V100` (Colab Pro)

---
## Step 0: Verify GPU is available

In [ ]:
# Check that Colab gave you a GPU
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

---
## Step 1: Clone & install ego2dex

In [ ]:
# Clone the repo
!git clone https://github.com/adityagarg7/ego2dex.git
%cd ego2dex

In [ ]:
# Install core (lightweight, no GPU models yet)
!pip install -e ".[dev]" -q

In [ ]:
# Verify the CLI works
!ego2dex info

---
## Step 2: Run the smoke test (CPU dry-run, no weights)

This validates the entire pipeline graph works using synthetic data.  
No GPU, no model downloads needed.

In [ ]:
# Run the smoke pipeline (CPU, dry-run, synthetic data)
!ego2dex run \
    --config configs/pipeline/smoke.yaml \
    --input assets/synthetic \
    --output outputs/smoke

In [ ]:
# Run the quickstart example (smoke + COCO export + ORCA retarget dry-run)
!python examples/quickstart.py

In [ ]:
# Check what was generated
!ls -la outputs/smoke/ 2>/dev/null || echo "Check outputs/quickstart/ instead"
!ls -la outputs/quickstart/ 2>/dev/null || true

---
## Step 3: Install GPU models for REAL video processing

Pick which models you need. Each one downloads weights on first use.

### Recommended stack for egocentric hand videos:
- **HaMeR** → 3D hand pose + MANO
- **Grounding DINO** → open-vocabulary object detection  
- **SAM 2** → segmentation + tracking
- **Qwen2.5-VL** → captions + tags
- **dex-retargeting** → retarget to robot hand

In [ ]:
# Install GPU model extras (this takes a few minutes)
# Uncomment the models you want:

# --- Hand Pose (pick one) ---
!pip install -e ".[hamer]" -q          # HaMeR (recommended, needs MANO)
# !pip install -e ".[mediapipe]" -q    # MediaPipe (lighter, CPU, no MANO)

# --- Object Detection + Segmentation ---
!pip install -e ".[detection]" -q      # Grounding DINO
!pip install git+https://github.com/facebookresearch/sam2.git -q  # SAM 2

# --- Captions / Tags ---
!pip install -e ".[caption]" -q        # Qwen2.5-VL, RAM++

# --- Retargeting + Export ---
!pip install -e ".[retarget,export]" -q  # dex-retargeting + HDF5/parquet

print("\n✅ GPU model extras installed!")

In [ ]:
# Install source-only models + download weights
# (HaMeR, SAM2 checkpoints, Grounding DINO)
!bash scripts/install_models.sh hamer sam2 grounding_dino

### ⚠️ MANO Setup (Required for HaMeR / any MANO-based hand model)

MANO is **research-only and gated**. ego2dex does NOT include it.

1. Register at https://mano.is.tue.mpg.de
2. Download `MANO_RIGHT.pkl` and `MANO_LEFT.pkl`
3. Upload them to Colab (next cell)

**If you don't have MANO**, you can still use **MediaPipe** (no MANO needed) — just change the config below.

In [ ]:
# Upload MANO files (skip if using MediaPipe)
import os
os.makedirs("_DATA/mano", exist_ok=True)

# Option A: Upload from your computer
from google.colab import files
print("Upload MANO_RIGHT.pkl and MANO_LEFT.pkl:")
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f"_DATA/mano/{name}", "wb") as f:
        f.write(data)
    print(f"  Saved: _DATA/mano/{name}")

# Option B: If you have them on Google Drive, uncomment:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/path/to/MANO_*.pkl _DATA/mano/

---
## Step 4: Upload YOUR egocentric videos

In [ ]:
import os
os.makedirs("my_videos", exist_ok=True)

# === Option A: Upload from your computer ===
from google.colab import files
print("Upload your egocentric video file(s) (.mp4, .mov, .avi):")
uploaded = files.upload()
for name, data in uploaded.items():
    filepath = f"my_videos/{name}"
    with open(filepath, "wb") as f:
        f.write(data)
    print(f"  ✅ Saved: {filepath} ({len(data)/1e6:.1f} MB)")

In [ ]:
# === Option B: Mount Google Drive (better for large videos) ===
# Uncomment and run this if your videos are on Google Drive

# from google.colab import drive
# drive.mount('/content/drive')

# # Copy your video(s) from Drive
# !cp /content/drive/MyDrive/path/to/your_video.mp4 my_videos/

# List uploaded videos
!ls -lh my_videos/

---
## Step 5: Process your video with the FULL pipeline 🚀

This runs the complete SOTA stack on your video:
- 3D hand pose (HaMeR + MANO)
- Object detection (Grounding DINO)
- Segmentation + tracking (SAM 2)
- Captions (Qwen2.5-VL)
- Export (JSON + COCO + HDF5)

In [ ]:
# ============================
# CONFIGURE YOUR RUN HERE
# ============================

VIDEO_FILE = "my_videos/your_video.mp4"  # <-- CHANGE THIS to your video filename
OUTPUT_DIR = "outputs/my_run"

# Check the file exists
import os
if os.path.exists(VIDEO_FILE):
    size_mb = os.path.getsize(VIDEO_FILE) / 1e6
    print(f"✅ Video found: {VIDEO_FILE} ({size_mb:.1f} MB)")
else:
    print(f"❌ Video not found: {VIDEO_FILE}")
    print(f"   Available files in my_videos/:")
    !ls my_videos/

In [ ]:
# === Option A: Full pipeline with HaMeR (needs MANO + GPU) ===
!ego2dex run \
    --config configs/pipeline/default.yaml \
    --input {VIDEO_FILE} \
    --output {OUTPUT_DIR}

In [ ]:
# === Option B: MediaPipe-only (NO MANO needed, lighter, works on free Colab) ===
# Use this if you don't have MANO files or want a quicker test

# !pip install -e ".[mediapipe]" -q
# !ego2dex run \
#     --config configs/pipeline/smoke.yaml \
#     --input {VIDEO_FILE} \
#     --output {OUTPUT_DIR}

---
## Step 6: Inspect the results

In [ ]:
# List all output files
import os
OUTPUT_DIR = "outputs/my_run"  # match the output dir from Step 5

for root, dirs, files_list in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files_list:
        filepath = os.path.join(root, file)
        size_kb = os.path.getsize(filepath) / 1024
        print(f'{subindent}{file} ({size_kb:.1f} KB)')

In [ ]:
# Load and inspect the clip annotation
import json
from pathlib import Path

clip_files = list(Path(OUTPUT_DIR).rglob("*.json"))
print(f"Found {len(clip_files)} JSON files:")
for f in clip_files:
    print(f"  {f}")

# Load the main clip annotation (if exists)
clip_path = Path(OUTPUT_DIR) / "clip_full.json"
if clip_path.exists():
    with open(clip_path) as f:
        clip_data = json.load(f)
    print(f"\n📊 Clip annotation summary:")
    print(f"  Frames:     {len(clip_data.get('frames', []))}")
    n_hands = sum(len(f.get('hands', [])) for f in clip_data.get('frames', []))
    n_dets = sum(len(f.get('detections', [])) for f in clip_data.get('frames', []))
    print(f"  Hands:      {n_hands}")
    print(f"  Detections: {n_dets}")
    print(f"  Retarget:   {len(clip_data.get('retargeting', []))} result(s)")
else:
    print(f"\nNo clip_full.json found. Check output files above.")

In [ ]:
# Visualize hand keypoints on a frame (if available)
from pathlib import Path
from IPython.display import display, Image as IPImage

viz_dir = Path(OUTPUT_DIR)
overlay_images = sorted(viz_dir.rglob("*.jpg")) + sorted(viz_dir.rglob("*.png"))

if overlay_images:
    print(f"Found {len(overlay_images)} output images. Showing first 5:")
    for img_path in overlay_images[:5]:
        print(f"\n{img_path.name}:")
        display(IPImage(filename=str(img_path), width=600))
else:
    print("No overlay images found. Run the viz command to generate them:")
    print(f'  ego2dex viz --clip {OUTPUT_DIR}/clip_full.json --frames <frames_dir> --output {OUTPUT_DIR}/viz')

---
## Step 7: Export for VLA training

In [ ]:
# Export to different formats for downstream training
CLIP_JSON = f"{OUTPUT_DIR}/clip_full.json"

# COCO format (keypoints + categories)
!ego2dex export --clip {CLIP_JSON} --writer coco --output outputs/export_coco

# JSON format
!ego2dex export --clip {CLIP_JSON} --writer json --output outputs/export_json

# HDF5 format (for imitation learning)
!ego2dex export --clip {CLIP_JSON} --writer hdf5 --output outputs/export_hdf5

---
## Step 8: ORCA retargeting (optional)

Map human hand motion → ORCA robot hand (16 DOF) joints.

For **dry-run** (no URDF needed):

In [ ]:
# Retarget to ORCA hand (dry-run — synthetic joint trajectories)
!ego2dex retarget \
    --clip {CLIP_JSON} \
    --robot orca \
    --dry-run \
    --output outputs/retarget_orca

# For LIVE retargeting (needs real URDF):
# import os
# os.environ['EGO2DEX_ORCA_URDF'] = '/path/to/orcahand.urdf'
# !ego2dex retarget --clip {CLIP_JSON} --robot orca --urdf $EGO2DEX_ORCA_URDF --live --output outputs/retarget_orca

---
## Step 9: Download results to your machine

In [ ]:
# Zip all outputs and download
!zip -r outputs_ego2dex.zip outputs/

from google.colab import files
files.download('outputs_ego2dex.zip')
print("\n✅ Download started! Check your browser downloads.")

In [ ]:
# Or save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r outputs/ /content/drive/MyDrive/ego2dex_outputs/

---
## 🔧 Troubleshooting

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | Reduce `io.sample_fps` or `io.max_frames` in the config, or use a smaller model (MediaPipe instead of HaMeR) |
| `MANO_RIGHT.pkl not found` | Upload MANO files (Step 3) or switch to MediaPipe config |
| `ModuleNotFoundError` for a stage | Install the relevant extra: `pip install -e ".[extra_name]"` |
| Kernel crashes | You may need Colab Pro (more RAM/VRAM). Try processing shorter clips. |
| Slow processing | Set `io.max_frames: 50` in your config to process fewer frames |

### Creating a custom config for Colab
If the default config is too heavy for free-tier Colab (T4 GPU, 15GB VRAM), create a lighter one:

In [ ]:
# Create a lighter Colab-friendly config
colab_config = """
# Colab-friendly pipeline: lighter than default, works on T4 GPU
name: colab_gopro
description: "Lighter GoPro pipeline for Colab (T4 GPU)."

io:
  sample_fps: 5              # lower FPS = fewer frames = less VRAM
  max_frames: 100            # cap total frames (remove for full video)
  source: gopro
  undistort: true

run:
  device: cuda
  dry_run: false
  strict: false
  output_dir: outputs/colab_run

mano:
  model_dir: _DATA/mano       # set to null if using MediaPipe

stages:
  # 3D hands (HaMeR) + smoothing
  - include: configs/hands/hamer.yaml
  - { family: hands, name: smoothing, params: { method: oneeuro } }
  # Open-vocab detection (Grounding DINO)
  - include: configs/detection/grounded_sam2.yaml
  # Export
  - include: configs/export/json.yaml
  - include: configs/export/coco.yaml
  - { family: viz, name: overlay }
"""

with open("configs/pipeline/colab.yaml", "w") as f:
    f.write(colab_config)

print("✅ Created configs/pipeline/colab.yaml")
print("   Use with: ego2dex run -c configs/pipeline/colab.yaml -i my_videos/your_video.mp4 -o outputs/colab_run")

---
## 📚 Quick Reference

```bash
# List all stages
ego2dex info

# Validate a config without running
ego2dex validate -c configs/pipeline/default.yaml

# Run pipeline
ego2dex run -c <config.yaml> -i <video.mp4> -o <output_dir>

# Export from existing annotations
ego2dex export --clip <clip.json> --writer coco|json|hdf5 -o <dir>

# Retarget to robot hand
ego2dex retarget --clip <clip.json> --robot orca|allegro|shadow|leap

# Render overlays
ego2dex viz --clip <clip.json> --frames <frames_dir> -o <viz_dir>
```